In [0]:
from pyspark.sql.functions import *

source_table = (
    "retail_project.external_app_schema.future_transactions"
)

progress_table = (
    "retail_project.external_app_schema.simulator_progress"
)

df_source = spark.table(source_table)
df_progress = spark.table(progress_table)

In [0]:
available_dates = (
    df_source
    .select("transaction_date")
    .distinct()
)

published_dates = (
    df_progress
    .select("transaction_date")
    .distinct()
)

unpublished_dates = (
    available_dates
    .join(
        published_dates,
        on="transaction_date",
        how="left_anti"
    )
)

In [0]:
next_row = (
    unpublished_dates
    .orderBy("transaction_date")
    .first()
)

if next_row is None:
    raise RuntimeError("No unpublished transaction dates remain.")

selected_date = next_row["transaction_date"].isoformat()

print(f"Selected date: {selected_date}")

In [0]:
dbutils.jobs.taskValues.set(
    key="selected_date",
    value=selected_date
)